
## **Introduction to AnnData**

*   **AnnData** is a core data structure designed for matrix-based biological datasets, especially single-cell omics data.  

*   It stores observations such as cells and variables such as genes in a compact and organized format.

*   It also supports metadata, sparse matrices, and scalable downstream analysis in one unified object.




## **Initializing AnnData**
* An AnnData object can be created from a dense or sparse matrix.

* In most single-cell applications, this matrix represents gene expression counts.

* The main data matrix is stored in adata.X.




In [ ]:
%pip install anndata scanpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.5 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## **Importing the Required Libraries**

The notebook then imports the packages needed for creating and managing the dataset.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
print(ad.__version__)

0.12.10


/tmp/ipykernel_11572/2319192822.py:5: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(ad.__version__)




*   `numpy` is used for generating numerical data.

*   `pandas` is used for metadata tables such as `.obs`
*   `anndata` provides the AnnData object itself.
*   `csr_matrix` from SciPy enables sparse storage, which is ideal for gene expression data.




## **Building a Simple AnnData Object**
A random sparse count matrix is created and wrapped inside an AnnData object.




In [ ]:
counts = csr_matrix(np.random.poisson(1, size=(100, 2000)), dtype=np.float32)
adata = ad.AnnData(counts)
adata

AnnData object with n_obs × n_vars = 100 × 2000

In [ ]:
adata.X


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 126568 stored elements and shape (100, 2000)>

* The matrix contains 100 rows and 2000 columns, representing cells and genes.
* A Poisson distribution is used to simulate count-like data.
* The main expression matrix is stored in `adata.X`
* Using a sparse matrix saves memory because many values in single-cell data are zero.


## **Naming Cells and Genes**
The notebook assigns readable identifiers to observations and variables.

In [ ]:
adata.obs_names = [f"Cell_{i:d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:d}" for i in range(adata.n_vars)]
print(adata.obs_names[:10])

Index(['Cell_0', 'Cell_1', 'Cell_2', 'Cell_3', 'Cell_4', 'Cell_5', 'Cell_6',
       'Cell_7', 'Cell_8', 'Cell_9'],
      dtype='object')


* `.obs_names` stores cell identifiers.
* `.var_names` stores gene identifiers.

## **Selecting Subsets of the Data**
A small subset of cells and genes is extracted using their names.



In [ ]:
adata[["Cell_1", "Cell_10"], ["Gene_5", "Gene_1900"]]


View of AnnData object with n_obs × n_vars = 2 × 2

* AnnData supports label-based slicing, similar to pandas.
* Subsetting keeps the selected rows and columns aligned across the object.
* By default, this usually returns a view, not a fully independent copy.


## **Attaching Cell-Level Annotations**
The notebook creates a simple cell-type annotation and stores it in `.obs`.


In [ ]:
ct = np.random.choice(["B", "T", "Monocyte"], size=(adata.n_obs,))
adata.obs["cell_type"] = pd.Categorical(ct)  # Categoricals are preferred for efficiency
adata.obs


,cell_type
Cell_0,T
Cell_1,B
Cell_2,T
Cell_3,T
Cell_4,B
...,...
Cell_95,Monocyte
Cell_96,B
Cell_97,B
Cell_98,B


In [ ]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'

* `.obs` is a pandas DataFrame that stores one row of metadata per cell.
* The new column `cell_type` assigns each cell a category.
* `pd.Categorical` is memory-efficient and useful for repeated labels such as cluster names or cell types.
* This metadata stays aligned with the main matrix during subsetting.


## **Filtering by Biological Annotation**
The notebook filters cells based on the cell-type metadata created earlier.


In [ ]:
bdata = adata[adata.obs.cell_type == "B"]
bdata

View of AnnData object with n_obs × n_vars = 40 × 2000
    obs: 'cell_type'

* This creates a subset containing only cells labeled as "B".
* Boolean filtering on `.obs` is a standard way to isolate biologically relevant groups.
* The resulting object keeps only the selected cells while preserving the gene dimension.


## **Storing Embeddings and Gene-Level Matrices**
The notebook adds multi-dimensional information to the object using `.obsm` and `.varm`.


In [ ]:
adata.obsm["X_umap"] = np.random.normal(0, 1, size=(adata.n_obs, 2))
adata.varm["gene_stuff"] = np.random.normal(0, 1, size=(adata.n_vars, 5))
adata.obsm

AxisArrays with keys: X_umap

In [ ]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    obsm: 'X_umap'
    varm: 'gene_stuff'

* `.obsm` stores observation-level matrices
* `.varm` stores variable-level matrices
* The dimensions must match the number of observations or variables, respectively.
* This design keeps structured analysis outputs attached to the same dataset.



## **Saving Miscellaneous Results**

Unstructured information is stored in the .uns slot.


In [ ]:
adata.uns["random"] = [1, 2, 3]
adata.uns

OrderedDict([('random', [1, 2, 3])])

* `.uns` is used for results that do not fit into row- or column-aligned tables.
* It can store dictionaries, lists, colors, plotting settings, and method parameters.
* Many Scanpy functions automatically save analysis information here.


## **Keeping Multiple Data Representations**
The notebook creates a transformed version of the matrix and stores it in .layers.


In [ ]:
adata.layers["log_transformed"] = np.log1p(adata.X)
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'

* `.layers` stores alternate versions of the main matrix.
* This is helpful when you want to keep raw counts and transformed data together.
* `np.log1p()` is commonly used in gene expression workflows to reduce skewness while preserving zeros.
* Layers help avoid overwriting the original data in `adata.X`



##  **Exporting the Matrix as a DataFrame**
The notebook converts the data to a pandas DataFrame.

In [ ]:
adata.to_df(layer="log_transformed")

,Gene_0,Gene_1,Gene_2,Gene_3,Gene_4,Gene_5,Gene_6,Gene_7,Gene_8,Gene_9,...,Gene_1990,Gene_1991,Gene_1992,Gene_1993,Gene_1994,Gene_1995,Gene_1996,Gene_1997,Gene_1998,Gene_1999
Cell_0,0.693147,1.098612,0.000000,0.693147,1.098612,0.000000,0.693147,0.693147,0.693147,0.000000,...,1.386294,0.693147,0.000000,1.791759,0.693147,1.098612,0.000000,0.000000,0.693147,0.693147
Cell_1,0.693147,0.000000,0.693147,1.386294,0.693147,1.098612,0.000000,1.386294,0.000000,0.000000,...,0.000000,0.000000,1.098612,0.000000,0.693147,0.693147,0.000000,0.693147,0.000000,0.000000
Cell_2,0.693147,0.000000,0.693147,0.693147,1.098612,0.000000,0.693147,0.000000,0.693147,1.386294,...,0.693147,0.693147,1.098612,1.386294,0.000000,0.000000,0.000000,0.693147,0.693147,0.000000
Cell_3,0.693147,1.098612,1.098612,1.098612,1.098612,1.098612,0.000000,0.000000,1.098612,0.693147,...,0.000000,1.098612,0.693147,0.000000,0.000000,0.693147,0.000000,0.000000,0.000000,1.386294
Cell_4,1.098612,0.000000,1.098612,0.000000,0.000000,0.000000,0.693147,1.098612,1.098612,0.693147,...,0.693147,0.000000,0.693147,1.098612,0.693147,0.693147,0.000000,0.000000,1.098612,0.693147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cell_95,0.000000,0.693147,0.693147,0.000000,0.000000,0.693147,1.386294,1.098612,1.098612,1.386294,...,0.000000,1.098612,1.098612,0.000000,1.098612,0.000000,0.693147,0.000000,0.693147,0.000000
Cell_96,0.693147,0.693147,0.000000,0.693147,0.000000,0.693147,0.693147,0.000000,1.098612,0.000000,...,0.000000,0.000000,0.000000,0.693147,0.693147,0.000000,0.000000,0.000000,0.693147,1.609438
Cell_97,0.693147,1.386294,0.000000,1.098612,0.000000,1.098612,0.693147,0.693147,0.693147,0.693147,...,0.000000,0.000000,0.693147,0.000000,0.693147,0.693147,1.098612,0.693147,0.000000,0.000000
Cell_98,0.000000,0.693147,0.693147,0.000000,0.000000,0.000000,0.693147,1.098612,1.098612,1.098612,...,1.098612,0.000000,0.000000,0.693147,0.693147,0.000000,0.000000,1.098612,0.000000,1.098612


* `.to_df()` returns a tabular view of the expression matrix.
* The `layer` argument allows you to choose which data representation to export.
* This is useful for quick inspection, downstream tabular analysis, or saving values for external tools.


##  **Saving the Dataset to Disk**

The AnnData object is written to an .h5ad file.

In [ ]:
adata.write('my_results.h5ad', compression="gzip")

In [ ]:
!h5ls 'my_results.h5ad'

/bin/bash: line 1: h5ls: command not found


* `.write()` saves the complete AnnData object, including metadata and layers.
* Compression reduces file size, which is helpful for larger datasets.
* The `h5ls` command is a shell utility for inspecting HDF5 files, but it may not be installed in all notebook environments.
* If h5ls is unavailable, the file can still be read normally with AnnData.


## **Understanding Metadata Alignment**
The notebook builds a new observation metadata table and reinitializes the AnnData object with it.

In [ ]:
obs_meta = pd.DataFrame({
        'time_yr': np.random.choice([0, 2, 4, 8], adata.n_obs),
        'subject_id': np.random.choice(['subject 1', 'subject 2', 'subject 4', 'subject 8'], adata.n_obs),
        'instrument_type': np.random.choice(['type a', 'type b'], adata.n_obs),
        'site': np.random.choice(['site x', 'site y'], adata.n_obs),
    },
    index=adata.obs.index,    # these are the same IDs of observations as above!
)

In [ ]:
adata = ad.AnnData(adata.X, obs=obs_meta, var=adata.var)

In [ ]:
print(adata)

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'


In [ ]:
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

* The metadata table is indexed using the same cell identifiers as the original object.
* Correct index alignment is essential when attaching `.obs` data.
* Rebuilding the object this way demonstrates how matrix data and metadata can be reassembled into a new AnnData object.
* The printed summary confirms that the new observation columns were added successfully.

## **Inspecting Small Slices of the Dataset**

The notebook extracts a few cells and genes to inspect a small part of the object.

In [ ]:
adata[:5, ['Gene_1', 'Gene_3']]

View of AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [ ]:
adata_subset = adata[:5, ['Gene_1', 'Gene_3']].copy()

* Small slices are useful for debugging and understanding the object structure.
* Using `.copy()` creates an independent object that can be safely modified.
* Without `.copy()`, a subset may remain a view of the parent object.

## **Views, Copies, and Modification Warnings**

The notebook demonstrates how modifying a view can trigger warnings.

In [ ]:
print(adata[:3, 'Gene_1'].X.toarray().tolist())
adata[:3, 'Gene_1'].X = [0, 0, 0]
print(adata[:3, 'Gene_1'].X.toarray().tolist())

[[2.0], [0.0], [0.0]]
[[0.0], [0.0], [0.0]]


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:630: FutureWarning: Setting element `.X` of view of `AnnData` object will obey copy-on-write semantics in the next minor release. 
  if self._handle_view_X_cow(value):
/tmp/ipykernel_11572/2822107891.py:2: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata[:3, 'Gene_1'].X = [0, 0, 0]


* A slice like `adata[:3, 'Gene_1']` may be a view into the original object.
* Changing `.X` on a view can affect the parent object and trigger warnings.


## **When a View Becomes a Real Object**

The notebook further shows what happens when metadata is edited on a subset.

In [ ]:
adata_subset = adata[:3, ['Gene_1', 'Gene_2']]

In [ ]:
adata_subset

View of AnnData object with n_obs × n_vars = 3 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [ ]:
adata_subset.obs['foo'] = range(3)

/tmp/ipykernel_11572/2955902014.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_subset.obs['foo'] = range(3)


In [ ]:
adata_subset

AnnData object with n_obs × n_vars = 3 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site', 'foo'

* Editing `.obs` on a view triggers an `ImplicitModificationWarning`.
* AnnData converts the view into a standalone object when necessary.
* This protects the original object from unintended metadata edits.


## **Conditional Metadata Queries**
The notebook uses metadata values to retrieve a subset of the observation table.

In [ ]:
adata[adata.obs.time_yr.isin([2, 4])].obs.head()

,time_yr,subject_id,instrument_type,site
Cell_2,2,subject 8,type b,site y
Cell_3,2,subject 1,type b,site x
Cell_5,4,subject 2,type a,site y
Cell_6,4,subject 1,type b,site x
Cell_7,4,subject 2,type b,site y


* `.isin()` makes it easy to filter observations by multiple values.
* This is useful for selecting time points, treatment groups, donors, or batches.
* The result is a quick preview of the matching metadata rows.


## **Working with Large Files in Backed Mode**
The notebook demonstrates partial reading of an .h5ad file using backed mode.



In [ ]:
adata = ad.read_h5ad('my_results.h5ad', backed='r')


In [ ]:
import anndata as ad

adata = ad.read_h5ad('my_results.h5ad', backed='r')
print(adata)


AnnData object with n_obs × n_vars = 100 × 2000 backed at 'my_results.h5ad'
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'


* `backed='r'` opens the dataset in read-only mode without loading everything into memory.
* This is useful for large files that would otherwise consume too much RAM.
* The printed summary shows that the object is now backed by a file on disk.
* Only the needed portions are accessed as required.

## **Checking Backed Status**
The notebook checks whether the object is backed and where the file is stored.

In [ ]:
adata.isbacked

True

In [ ]:
adata.filename

PosixPath('my_results.h5ad')

In [ ]:
adata.file.close()

* `adata.isbacked` tells you whether the object is currently using file-backed mode.
* `adata.filename` returns the path of the .h5ad file.
* `adata.file.close()` is good practice when you are finished working with a backed object.
* Closing the file releases resources cleanly.